# 电信客户流失预测与客户细分

本Notebook核心流程与scripts/churn_analysis.py保持一致：
- 数据读取与校验
- 数据清洗与特征工程
- 分类模型训练与评估（ROC/PR）
- 客户分群（KMeans/DBSCAN）
- 结果文件导出

> 说明：若环境缺少可选依赖（如 imblearn、xgboost、lightgbm、shap），脚本会优先自动降级，保证主流程可运行。

In [ ]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_PATH = ROOT / 'data' / 'raw' / 'WA_Fn-UseC_-Telco-Customer-Churn.csv'

print('Project root:', ROOT)
print('Dataset exists:', DATA_PATH.exists())

if DATA_PATH.exists():
    raw = pd.read_csv(DATA_PATH)
    print('Raw shape:', raw.shape)
    print('Columns:', list(raw.columns[:8]), '...')

## 运行完整分析流程

此单元直接调用项目脚本，自动生成模型、图表与分群结果到 `output/` 和 `models/`。

In [ ]:
import sys

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from scripts.churn_analysis import main

main()

In [ ]:
# 查看核心输出结果
out_dir = ROOT / 'output'
metrics_path = out_dir / 'model_metrics.csv'
profile_path = out_dir / 'data_profile.json'

if metrics_path.exists():
    display(pd.read_csv(metrics_path))
else:
    print('model_metrics.csv not found, please run previous cell first.')

if profile_path.exists():
    print(json.dumps(json.loads(profile_path.read_text(encoding='utf-8')), ensure_ascii=False, indent=2))

## 已运行结果说明（基于当前项目输出）

以下结果来自本项目 `output/` 中已生成文件（最近一次成功运行）：

### 1) 数据概况
- 原始数据规模：`7043 x 21`
- 清洗后数据规模：`7043 x 27`
- 标签分布：流失 `1869`，未流失 `5174`
- 缺失值：清洗后 `0`

### 2) 分类模型对比（测试集）
- `logistic_regression`：Accuracy=0.7353，F1=0.6119，ROC-AUC=0.8423，PR-AUC=0.6324
- `random_forest`：Accuracy=0.7779，F1=0.5843，ROC-AUC=0.8220，PR-AUC=0.6177

> 按 ROC-AUC 排序，当前最佳模型为 `logistic_regression`。

### 3) 客户细分（KMeans）
- 聚类簇 0：平均 tenure≈19.82，平均月费≈51.53，流失率≈31.90%
- 聚类簇 1：平均 tenure≈55.22，平均月费≈88.83，流失率≈16.77%

根据结果进行解读：簇0客户在网时长短、消费低，流失风险更高；簇1客户在网时长长、消费高，稳定性更好。

In [ ]:
# 结果文件展示（指标表 + 聚类汇总 + 曲线图）
from IPython.display import Image, display

out_dir = ROOT / 'output'

metrics_df = pd.read_csv(out_dir / 'model_metrics.csv')
cluster_df = pd.read_csv(out_dir / 'kmeans_cluster_summary.csv')

print('模型指标：')
display(metrics_df)

print('KMeans 聚类汇总：')
display(cluster_df)

print('ROC 曲线：')
display(Image(filename=str(out_dir / 'roc_curves.png')))

print('PR 曲线：')
display(Image(filename=str(out_dir / 'pr_curves.png')))

print('聚类可视化：')
display(Image(filename=str(out_dir / 'clustering_scatter.png')))

## 结果图展示

### ROC 曲线
![ROC曲线](../output/roc_curves.png)

### PR 曲线
![PR曲线](../output/pr_curves.png)

### 客户分群可视化（KMeans / DBSCAN）
![客户分群可视化](../output/clustering_scatter.png)
